ARTI308 - Machine Learning

# Credit Card Customer Segmentation Project

In this project, you will use K-Means clustering to segment [credit card customers](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata/data) based on their usage behavior. This is an unsupervised learning problem because the dataset does not contain a target label for customer groups.

You will use the `CC_GENERAL.csv` dataset.

## About the Dataset

The dataset contains customer-level credit card usage behavior. Each row represents one credit card holder, and the columns describe different behavioral variables such as balance, purchases, cash advance, payments, and tenure. The goal is to group similar customers together so that the company can understand different customer segments and design better marketing strategies.

## Import Libraries

**Import the libraries you need for data analysis, visualization, preprocessing, clustering, and evaluation.**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")


## Get the Data

**Read the `CC_GENERAL.csv` file and save it in a dataframe called `df`.**

In [4]:
df = pd.read_csv(r"C:\Users\abdul\Desktop\Lab 11 ML\CC_GENERAL.csv")
df

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.40,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.00,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.00,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.00,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.00,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8945,C19186,28.493517,1.000000,291.12,0.00,291.12,0.000000,1.000000,0.000000,0.833333,0.000000,0,6,1000.0,325.594462,48.886365,0.500000,6
8946,C19187,19.183215,1.000000,300.00,0.00,300.00,0.000000,1.000000,0.000000,0.833333,0.000000,0,6,1000.0,275.861322,NaN,0.000000,6
8947,C19188,23.398673,0.833333,144.40,0.00,144.40,0.000000,0.833333,0.000000,0.666667,0.000000,0,5,1000.0,81.270775,82.418369,0.250000,6
8948,C19189,13.457564,0.833333,0.00,0.00,0.00,36.558778,0.000000,0.000000,0.000000,0.166667,2,0,500.0,52.549959,55.755628,0.250000,6


**Check the first five rows of the dataset.**

In [ ]:
df.head() 

**Check the shape of the dataset.**

In [ ]:
df.shape 

**Check basic information about the dataset using `info()`.**

In [ ]:
df.info()

**Check summary statistics using `describe()`.**

In [ ]:
df.describe()

## Data Cleaning

The column `CUST_ID` is an identification column. It is not useful for clustering because it does not describe customer behavior.

**Drop the `CUST_ID` column from the dataframe.**

In [ ]:
df = df.drop("CUST_ID", axis=1)
df.head()

**Check the missing values in each column.**

In [ ]:
df.isnull().sum()

Some columns may contain missing values.

Hint: You can handle missing values by either:
- filling them with the mean value
- or dropping the rows that contain missing values

For this project, use mean imputation.

**Fill the missing values with the mean of each column.**

In [ ]:
df = df.fillna(df.mean())

**Check the missing values again to make sure they were handled.**

In [ ]:
df.isnull().sum()

## Exploratory Data Analysis

Before applying clustering, it is important to understand the data.

**Create histograms for the numerical columns.**

In [ ]:
df.hist(figsize=(15, 12), bins=30)
plt.tight_layout()
plt.show()

**Create a correlation heatmap to understand relationships between the features.**

In [ ]:
plt.figure(figsize=(14, 10))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

**Create a scatter plot between `BALANCE` and `PURCHASES`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["BALANCE"], df["PURCHASES"])
plt.xlabel("BALANCE")
plt.ylabel("PURCHASES")
plt.title("BALANCE vs PURCHASES")
plt.show() 

**Create a scatter plot between `BALANCE` and `CASH_ADVANCE`.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["BALANCE"], df["CASH_ADVANCE"])
plt.xlabel("BALANCE")
plt.ylabel("CASH_ADVANCE")
plt.title("BALANCE vs CASH_ADVANCE")
plt.show() 

## Feature Scaling

K-Means is a distance-based algorithm. Therefore, feature scaling is very important.

The features in this dataset have very different ranges. For example, `BALANCE`, `PURCHASES`, and `CREDIT_LIMIT` may have large values, while frequency columns are between 0 and 1.

**Use StandardScaler to scale the data. Save the scaled data in a variable called `X_scaled`.**

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

## Choosing K Intuitively

Choosing K is one of the most difficult parts of K-Means.

Since this dataset has many features, it is not easy to visually see the clusters directly.

However, we can still compare different K values using the elbow method and silhouette score.

## Elbow Method

**Create a loop that fits K-Means models for K values from 1 to 10. Save the inertia values in a list called `inertia_values`.**

In [ ]:
inertia_values = []

for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

**Plot the elbow curve.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(1, 11), inertia_values, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

**Output Interpretation**

Look at the elbow curve and try to identify where the decrease in inertia starts to slow down.

That point can suggest a reasonable value for K.

## Silhouette Score

The silhouette score helps evaluate how well-separated the clusters are.

**Create a loop that calculates the silhouette score for K values from 2 to 10. Save the scores in a list called `silhouette_scores`.**

In [ ]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

**Plot the silhouette scores.**

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(range(2, 11), silhouette_scores, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Scores for Different K Values")
plt.show()

**Create a table showing each K value and its silhouette score.**

In [ ]:
silhouette_table = pd.DataFrame({
    "K": range(2, 11),
    "Silhouette Score": silhouette_scores
})

silhouette_table

**Output Interpretation**

A higher silhouette score usually means better clustering.

However, do not rely only on the highest value. Also consider whether the chosen K makes sense for customer segmentation.

## Create the Final K-Means Model

**Based on the elbow curve and silhouette scores, choose a final K value. Then train a final K-Means model.**

Use `random_state=42` and `n_init=10`.

In [ ]:
final_k = 4

kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
cluster_labels = kmeans_final.fit_predict(X_scaled) 

**Add the final cluster labels to the original dataframe in a new column called `Cluster`.**

In [ ]:
df["Cluster"] = cluster_labels 

**Check the first five rows after adding the cluster labels.**

In [ ]:
df.head()

## Cluster Analysis

Now we need to understand what each cluster means.

**Create a summary table using `groupby()` to show the mean values of each feature for each cluster.**

In [ ]:
cluster_summary = df.groupby("Cluster").mean()
cluster_summary

**Check how many customers are in each cluster.**

In [ ]:
df["Cluster"].value_counts() 

## Visualizing the Final Clusters

Since the dataset has many features, we will use PCA to reduce the data into two components only for visualization.

This visualization does not replace the original clustering. It only helps us see the clusters in a 2D plot.

**Use PCA with 2 components and plot the clusters.**

In [ ]:
pca = PCA(n_components=2)
pca_data = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(pca_data, columns=["PC1", "PC2"])
pca_df["Cluster"] = df["Cluster"]

plt.figure(figsize=(8, 6))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Cluster", palette="viridis")
plt.title("Final Clusters Visualized with PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.show()

**Output Interpretation**

The PCA plot gives a simplified 2D view of the clusters.

If the clusters are not perfectly separated, that is normal because the original dataset has many features and the plot only shows two compressed dimensions.

## Final Questions

Answer the following questions:

1. Why is this an unsupervised learning problem?

Because the dataset does not have a target/output label that tells us the correct customer group.

2. Why did we remove the `CUST_ID` column?

We removed CUST_ID because it is only an identification column.

3. Which columns had missing values?

CREDIT_LIMIT
MINIMUM_PAYMENTS    

4. How did you handle the missing values?

We used mean imputation, meaning each missing value was replaced with the mean value of it's column.


5. Why is scaling important before applying K-Means?

Because K-means uses distance to form clusters. By scaling, all features are put on a similar scale, so each column contributes more fairly to the clustering process.

6. Which K value did you choose? Explain your answer using the elbow method and silhouette score.

I chose K = 4

Using the elbow method, the intertia decreases quickly at first, then improvement starts to slow down around K = 4. This means adding clusters after 4 does not reduce the error as much.

Using the silhouette score, we compare how well-seperated the clusters are for different K values. A higher silhouette score means better cluster seperation.

7. Based on the cluster summary table, describe each customer segment in your own words.

Cluster 0 - Regular purchase customers
Balance and moderate purchases, using mostly credit cards for purchases.

Cluster 1 - High-value active spenders
Highest purchases, high credit limits, and high payments. These customers are the most valuable segment for the bank.

Cluster 2 - Cash Advance / high balance customers
This cluster group has very high balances and very high cash amounts, but lower purchases. Low full payment percentage, meaning more debt.

Cluster 3 - Low activity customers
This group has low purchases, low transaction frequency, and lower credit limits.

8. Which cluster may represent high-value customers?

Cluster 1

9. Which cluster may represent customers who rely more on cash advance?

Cluster 2

10. How can a company use these clusters for marketing strategy?

Instead of sending the same offer to every customer, a company can use targeted marketing strategies.
